In [2]:
import pandas as pd
import os
from collections import Counter

def analisar_padroes_textuais():
    diretorio = os.path.join(os.getcwd(), "DADOS")
    
    if not os.path.exists(diretorio):
        print(f"Erro: Pasta {diretorio} não encontrada.")
        return None, None, None

    # Mapear todos os arquivos XLSX
    lista_excel = []
    for pasta in sorted(os.listdir(diretorio)):
        caminho_pasta = os.path.join(diretorio, pasta)
        if os.path.isdir(caminho_pasta):
            for arquivo in os.listdir(caminho_pasta):
                if arquivo.endswith('.xlsx') and not arquivo.startswith('~'):
                    lista_excel.append(os.path.join(caminho_pasta, arquivo))

    print(f"Analisando texto em {len(lista_excel)} arquivos...\n")
    
    # Contadores separados para cada categoria
    counter_estabelecimentos = Counter()
    counter_produtos = Counter()
    counter_marcas = Counter()

    # Varrer os arquivos
    for arquivo in lista_excel:
        df_planilha = pd.read_excel(arquivo, index_col=[1], sheet_name=None)
        
        for nome_aba, df in df_planilha.items():
            nome_aba_str = str(nome_aba).strip()
            
            list_index = df.index.unique().tolist()
            if pd.isna(list_index[0]):
                list_index.pop(0)
                
            for idx in list_index:
                idx_str = str(idx).strip()
                
                for coluna in df.columns:
                    celula = df.loc[idx, coluna]
                    
                    if hasattr(celula, 'values'):
                        celula = celula.values
                    elif not isinstance(celula, (list, tuple)):
                        celula = [celula]
                    
                    if len(celula) >= 2:
                        valor_nome = celula[0]
                        preco = celula[1]
                        
                        # Ignora células vazias ou marcadas como não coletadas
                        if pd.isna(preco) or str(preco).strip().lower() == 'xxx':
                            continue
                        
                        valor_nome_str = str(valor_nome).strip() if pd.notna(valor_nome) else "VAZIO/NAN"
                        
                        # Aplica a regra de negócio do seu projeto
                        if nome_aba_str.upper() == 'COMBUSTÍVEL':
                            # Para combustível: a aba é "Combustível", o index é o Posto, a coluna 0 é o Produto
                            counter_estabelecimentos[idx_str] += 1
                            counter_produtos[valor_nome_str] += 1
                            # Não contabilizamos marca aqui pois assumimos "Sem marca"
                        else:
                            # Para mercado: a aba é o Mercado, o index é o Produto, a coluna 0 é a Marca
                            counter_estabelecimentos[nome_aba_str] += 1
                            counter_produtos[idx_str] += 1
                            counter_marcas[valor_nome_str] += 1

    # Converter os contadores em DataFrames ordenados
    df_estabelecimentos = pd.DataFrame(
        counter_estabelecimentos.items(), columns=['Estabelecimento (Original)', 'Frequência']
    ).sort_values(by='Frequência', ascending=False).reset_index(drop=True)
    
    df_produtos = pd.DataFrame(
        counter_produtos.items(), columns=['Produto (Original)', 'Frequência']
    ).sort_values(by='Frequência', ascending=False).reset_index(drop=True)
    
    df_marcas = pd.DataFrame(
        counter_marcas.items(), columns=['Marca (Original)', 'Frequência']
    ).sort_values(by='Frequência', ascending=False).reset_index(drop=True)
    
    return df_estabelecimentos, df_produtos, df_marcas

# Executa a função
df_estab, df_prod, df_marc = analisar_padroes_textuais()

# Exibe os resultados (se estiver no Jupyter Notebook)
print("--- ESTABELECIMENTOS ---")
display(df_estab)

print("\n--- PRODUTOS ---")
display(df_prod)

print("\n--- MARCAS ---")
display(df_marc)

Analisando texto em 46 arquivos...

--- ESTABELECIMENTOS ---


,Estabelecimento (Original),Frequência
0,SUPER MAAR,9589
1,NACIONAL,9558
2,BOM PREÇO,9301
3,CARIBÉ,7830
4,PAG POUCO,7823
...,...,...
63,POSTO JB COMBUSTIVEIS (1L),14
64,POSTO JB COMBUSTÍVEIS 2 (1L),7
65,POSTO ALVORADA II (1L),7
66,POSTO ITAPIRAÇABA (1L),7



--- PRODUTOS ---


,Produto (Original),Frequência
0,ARROZ (5kg),5044
1,MACARRÃO (pac. 500g),4311
2,CARNE DE PRIMEIRA (1kg),4242
3,LEITE (1L),4238
4,FARINHA DE TRIGO (1kg),4220
5,CARNE DE SEGUNDA (1kg),4210
6,FEIJÃO (1kg),4209
7,MARGARINA (250g),4202
8,CAFÉ / em pó (500g),4201
9,PAPEL HIGIÊNICO (pac. 4 unid.),4188



--- MARCAS ---


,Marca (Original),Frequência
0,VAZIO/NAN,89452
1,Ypê,1578
2,Sadia,1437
3,Codil,1224
4,Frial,1200
...,...,...
400,coxão duro,1
401,Nova Espera,1
402,Norte Min.,1
403,flamboyant,1


In [5]:
#Verifica se a pasta existe; se não, cria.
os.makedirs('analise', exist_ok=True)

with pd.ExcelWriter('analise/analise_padrao.xlsx') as writer:
    df_estab.to_excel(writer, sheet_name='padrao_estabelecimentos', index=False)
    df_prod.to_excel(writer, sheet_name='padrao_produto', index=False)
    df_marc.to_excel(writer, sheet_name='padrao_marca', index=False)